# Hello VLA

Welcome to your very first hands on lesson. In this notebook you will:

1. Confirm your Colab runtime is working.
2. Load a real multimodal model (SmolVLM) and ask it to describe an image.
3. Try to load a real Vision-Language-Action model (SmolVLA) and inspect it.

Run each cell in order by pressing **Shift + Enter**.

## Step 1. Turn on a free GPU (optional)

Click **Runtime** then **Change runtime type**, pick **T4 GPU**, then **Save**. If no GPU is available, CPU still works, just slower.

In [ ]:
import torch

device = 0 if torch.cuda.is_available() else -1
print('Using device:', 'cuda' if device == 0 else 'cpu')
print('PyTorch version:', torch.__version__)

## Step 2. Ask SmolVLM to describe an image

A VLA is made of two halves: a **vision language model** that understands images and text, and an **action expert** that turns that understanding into robot actions. We are going to start by running the vision language half on its own, using Hugging Face's high level `pipeline` API which handles all the loading details for us.

In [ ]:
!pip install -q -U transformers

In [ ]:
from transformers import pipeline

model_id = 'HuggingFaceTB/SmolVLM-500M-Instruct'
dtype = torch.bfloat16 if device == 0 else torch.float32

vlm = pipeline(
    task='image-text-to-text',
    model=model_id,
    dtype=dtype,
    device=device,
)
print('Loaded', model_id)

In [ ]:
from PIL import Image
import requests

image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/640px-Cat03.jpg'
headers = {'User-Agent': 'vla-course/1.0'}
image = Image.open(requests.get(image_url, headers=headers, stream=True).raw).convert('RGB')
image.thumbnail((512, 512))
image

In [ ]:
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': 'Describe this scene in one sentence, then list three objects you can see.'},
        ],
    }
]

result = vlm(text=messages, max_new_tokens=120)
print(result[0]['generated_text'][-1]['content'])

Stop and notice what just happened. You gave a neural network an image plus a question in English, and it answered in English. This same kind of model is the brain of every modern VLA. The only piece missing is a small network that turns the understanding into joint angles. That piece is called the **action expert**, and together they form a Vision-Language-Action model.

## Step 3. Peek inside a real VLA (optional)

The cell below tries to install LeRobot and load **SmolVLA**, the actual 450M parameter VLA from the Hugging Face LeRobot team. If the install conflicts with something in today's Colab environment, you will get a clean message instead of a crash. Either outcome is fine. The main point of this module is already done.

In [ ]:
!pip install -q 'lerobot[smolvla]' 2>&1 | tail -n 5

In [ ]:
try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    policy = SmolVLAPolicy.from_pretrained('lerobot/smolvla_base')
    n_params = sum(p.numel() for p in policy.parameters())
    print('Loaded lerobot/smolvla_base')
    print(f'Total parameters: {n_params/1e6:.1f}M')
    print()
    print('Top level modules inside SmolVLA:')
    for name, _ in policy.named_children():
        print(' -', name)
except Exception as e:
    print('SmolVLA did not load cleanly in this environment.')
    print('That is OK. You already saw the vision language half work above.')
    print()
    print('Error was:')
    print(repr(e))

## What you just did

1. You ran a 500M parameter multimodal model on an image and got an English answer.
2. You saw, with your own eyes, that the vision language brain of a VLA is a file you can download and run in a browser.
3. You optionally loaded the full SmolVLA and inspected its architecture.

In the next modules we walk back to the start and build up every piece of what you just ran, from NumPy and gradient descent, to the Transformer, to modern robotics math, all the way back to fine tuning a VLA.

See you in Module 1.